# 21 · Réseaux convolutifs (CNN) — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch21_cnn/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch21_cnn/03_notebook.ipynb`, créée par `python tools/start_chapter.py 21`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 21.Q1 à 21.Q12, 21.R1 à 21.R3, 21.1 à 21.6, 21.8 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★★ | |
| A | 21.12 à 21.18 | ta librairie conv.py : tailles et padding, la vue « œil de mouche » (im2col), la convolution et le pooling, des tests paramétrés, la chasse aux rayures (*prochaine session de génération*) | | | |
| B | 21.19 à 21.21 | des filtres faits main avec PyTorch, convolutions 1D et 1 × 1, le détecteur de visage hiérarchique (*prochaine session de génération*) | | | |
| C | 21.22 à 21.26 | le convnet MNIST du livre sur Fashion-MNIST, ses courbes, une chasse aux bugs de formes, l'initialisation, stride ou pooling sur CIFAR-10 (*prochaine session de génération*) | | | |
| D | 21.27 à 21.32 | les filtres d'un ResNet-18 pré-entraîné, la montée de gradient sur l'entrée, l'attaque FGSM, la convolution transposée et la rétropropagation dans ta librairie, le défi Fashion-MNIST (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="21")  # your own library

## Objectifs et rappel express

- Calculer à la main la taille de sortie, le nombre de paramètres et le champ récepteur d'une pile de convolutions et de poolings.
- Programmer en NumPy, dans `mylearn.conv`, la convolution, `im2col`, le pooling, la convolution transposée et la rétropropagation, et les comparer à `torch.nn.functional`.
- Entraîner des CNN PyTorch, regarder leurs filtres et les tromper par une attaque adversariale.

**Rappel express.** Une couche de convolution calcule une corrélation croisée : $y_{n,o,i,j} = b_o + \sum_{c,u,v} W_{o,c,u,v}\,x_{n,c,\,s i + u,\,s j + v}$, poids de forme $(C_{out}, C_{in}, k_h, k_w)$, $C_{out}(C_{in}k_hk_w + 1)$ paramètres. Taille de sortie : $\lfloor (H + 2p - d(k-1) - 1)/s \rfloor + 1$ ; convolution transposée : $(H - 1)s - 2p + k$. Champ récepteur : $r_\ell = r_{\ell-1} + (k_\ell - 1)\prod_{i<\ell} s_i$. FGSM : $\mathbf{x} + \varepsilon\,\mathrm{sign}(\nabla_{\mathbf{x}}L)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 21.1 à 21.6 et 🧮 21.8)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme ou une position : entre guillemets (`"(3, 5)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 21.7, la réflexion (🗣️ 21.9, ⚖️ 21.10, 📄 21.11) et l'entretien se corrigent avec `05_solutions.md`.

**21.Q1 — Les deux sens de « profondeur »**

In [ ]:
# 21.Q1: the values you found on paper
answer_21_Q1a = ...  # a) a whole number
answer_21_Q1b = ...  # b) a whole number
answer_21_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_21_Q1d = ...  # d) a whole number
answer_21_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_21_Q1a, answer_21_Q1b, answer_21_Q1c, answer_21_Q1d, answer_21_Q1e]):
    wb.check(f"21.Q1{letter}", answer)

**21.Q2 — Un seul neurone promené sur toute l'image**

In [ ]:
# 21.Q2: the values you found on paper
answer_21_Q2a = ...  # a) True or False
answer_21_Q2b = ...  # b) a whole number
answer_21_Q2c = ...  # c) True or False
answer_21_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_21_Q2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_21_Q2a, answer_21_Q2b, answer_21_Q2c, answer_21_Q2d, answer_21_Q2e]):
    wb.check(f"21.Q2{letter}", answer)

**21.Q3 — Convolution ou corrélation croisée ?**

In [ ]:
# 21.Q3: the values you found on paper
answer_21_Q3a = ...  # a) True or False
answer_21_Q3b = ...  # b) a whole number
answer_21_Q3c = ...  # c) a whole number
answer_21_Q3d = ...  # d) True or False
answer_21_Q3e = ...  # e) a whole number
answer_21_Q3f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_21_Q3a, answer_21_Q3b, answer_21_Q3c, answer_21_Q3d, answer_21_Q3e, answer_21_Q3f]):
    wb.check(f"21.Q3{letter}", answer)

**21.Q4 — Détecteurs de motifs et hiérarchie de filtres**

In [ ]:
# 21.Q4: the values you found on paper
answer_21_Q4a = ...  # a) a whole number
answer_21_Q4b = ...  # b) a whole number
answer_21_Q4c = ...  # c) True or False
answer_21_Q4d = ...  # d) the letter of your choice, e.g. "E"
answer_21_Q4e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_21_Q4a, answer_21_Q4b, answer_21_Q4c, answer_21_Q4d, answer_21_Q4e]):
    wb.check(f"21.Q4{letter}", answer)

**21.Q5 — Padding et stride : que devient la taille ?**

In [ ]:
# 21.Q5: the values you found on paper
answer_21_Q5a = ...  # a) a whole number
answer_21_Q5b = ...  # b) a whole number
answer_21_Q5c = ...  # c) a whole number
answer_21_Q5d = ...  # d) a whole number
answer_21_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_21_Q5a, answer_21_Q5b, answer_21_Q5c, answer_21_Q5d, answer_21_Q5e]):
    wb.check(f"21.Q5{letter}", answer)

**21.Q6 — Canaux d'entrée, nombre de filtres, canaux de sortie**

In [ ]:
# 21.Q6: the values you found on paper
answer_21_Q6a = ...  # a) a whole number
answer_21_Q6b = ...  # b) a whole number
answer_21_Q6c = ...  # c) a whole number
answer_21_Q6d = ...  # d) a whole number
answer_21_Q6e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_21_Q6a, answer_21_Q6b, answer_21_Q6c, answer_21_Q6d, answer_21_Q6e]):
    wb.check(f"21.Q6{letter}", answer)

**21.Q7 — Convolution 1D ou convolution 1×1 ?**

In [ ]:
# 21.Q7: the values you found on paper
answer_21_Q7a = ...  # a) a whole number
answer_21_Q7b = ...  # b) a whole number
answer_21_Q7c = ...  # c) a whole number
answer_21_Q7d = ...  # d) True or False
answer_21_Q7e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_21_Q7a, answer_21_Q7b, answer_21_Q7c, answer_21_Q7d, answer_21_Q7e]):
    wb.check(f"21.Q7{letter}", answer)

**21.Q8 — Couche de convolution et initialisation des filtres**

In [ ]:
# 21.Q8: the values you found on paper
answer_21_Q8a = ...  # a) True or False
answer_21_Q8b = ...  # b) a whole number
answer_21_Q8c = ...  # c) a number, 3 decimals
answer_21_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_21_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_21_Q8a, answer_21_Q8b, answer_21_Q8c, answer_21_Q8d, answer_21_Q8e]):
    wb.check(f"21.Q8{letter}", answer)

**21.Q9 — Convolution transposée : les mots piégés**

In [ ]:
# 21.Q9: the values you found on paper
answer_21_Q9a = ...  # a) True or False
answer_21_Q9b = ...  # b) a whole number
answer_21_Q9c = ...  # c) a whole number
answer_21_Q9d = ...  # d) a whole number
answer_21_Q9e = ...  # e) True or False
answer_21_Q9f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_21_Q9a, answer_21_Q9b, answer_21_Q9c, answer_21_Q9d, answer_21_Q9e, answer_21_Q9f]):
    wb.check(f"21.Q9{letter}", answer)

**21.Q10 — VGG16 et ce que voient ses filtres**

In [ ]:
# 21.Q10: the values you found on paper
answer_21_Q10a = ...  # a) a shape as a string, e.g. "(3, 5)"
answer_21_Q10b = ...  # b) a whole number
answer_21_Q10c = ...  # c) a whole number
answer_21_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_21_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_21_Q10a, answer_21_Q10b, answer_21_Q10c, answer_21_Q10d, answer_21_Q10e]):
    wb.check(f"21.Q10{letter}", answer)

**21.Q11 — Exemples adversariaux : vrai ou faux ?**

In [ ]:
# 21.Q11: the values you found on paper
answer_21_Q11a = ...  # a) True or False
answer_21_Q11b = ...  # b) True or False
answer_21_Q11c = ...  # c) True or False
answer_21_Q11d = ...  # d) a number, 2 decimals
answer_21_Q11e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_21_Q11a, answer_21_Q11b, answer_21_Q11c, answer_21_Q11d, answer_21_Q11e]):
    wb.check(f"21.Q11{letter}", answer)

**21.Q12 — 🕰️ Après VGG16 : ResNet, ViT et modèles pré-entraînés**

In [ ]:
# 21.Q12: the values you found on paper
answer_21_Q12a = ...  # a) the letter of your choice, e.g. "E"
answer_21_Q12b = ...  # b) a whole number
answer_21_Q12c = ...  # c) a whole number
answer_21_Q12d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_21_Q12a, answer_21_Q12b, answer_21_Q12c, answer_21_Q12d]):
    wb.check(f"21.Q12{letter}", answer)

**21.R1 — Ce que changent model.train() et model.eval() pour le dropout**

In [ ]:
# 21.R1: the values you found on paper
answer_21_R1a = ...  # a) True or False
answer_21_R1b = ...  # b) a number, 4 decimals
answer_21_R1c = ...  # c) a whole number (%)
answer_21_R1d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_21_R1a, answer_21_R1b, answer_21_R1c, answer_21_R1d]):
    wb.check(f"21.R1{letter}", answer)

**21.R2 — Le gradient d'un poids utilisé à plusieurs endroits**

In [ ]:
# 21.R2: the values you found on paper
answer_21_R2a = ...  # a) a whole number
answer_21_R2b = ...  # b) a whole number
answer_21_R2c = ...  # c) True or False

for letter, answer in zip("abc", [answer_21_R2a, answer_21_R2b, answer_21_R2c]):
    wb.check(f"21.R2{letter}", answer)

**21.R3 — Le produit scalaire, ce que calcule un filtre en chaque position**

In [ ]:
# 21.R3: the values you found on paper
answer_21_R3a = ...  # a) a whole number
answer_21_R3b = ...  # b) a whole number
answer_21_R3c = ...  # c) True or False

for letter, answer in zip("abc", [answer_21_R3a, answer_21_R3b, answer_21_R3c]):
    wb.check(f"21.R3{letter}", answer)

**Ex 21.1 — Le détecteur de jaune à la main**

In [ ]:
# Ex 21.1: the values you found on paper
answer_21_1a = ...  # a) a number
answer_21_1b = ...  # b) a number
answer_21_1c = ...  # c) a number
answer_21_1d = ...  # d) a number
answer_21_1e = ...  # e) the letter of your choice, e.g. "E"
answer_21_1f = ...  # f) a number
answer_21_1g = ...  # g) a whole number
answer_21_1h = ...  # h) a whole number
answer_21_1i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_21_1a, answer_21_1b, answer_21_1c, answer_21_1d, answer_21_1e, answer_21_1f, answer_21_1g, answer_21_1h, answer_21_1i]):
    wb.check(f"21.1{letter}", answer)

**Ex 21.2 — Une corrélation 5×5 par 3×3 à la main**

In [ ]:
# Ex 21.2: the values you found on paper
answer_21_2a = ...  # a) a whole number
answer_21_2b = ...  # b) a whole number
answer_21_2c = ...  # c) a whole number
answer_21_2d = ...  # d) a whole number
answer_21_2e = ...  # e) a whole number
answer_21_2f = ...  # f) a position as a string, e.g. "(0, 4)"
answer_21_2g = ...  # g) a whole number
answer_21_2h = ...  # h) a whole number
answer_21_2i = ...  # i) a whole number
answer_21_2j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_21_2a, answer_21_2b, answer_21_2c, answer_21_2d, answer_21_2e, answer_21_2f, answer_21_2g, answer_21_2h, answer_21_2i, answer_21_2j]):
    wb.check(f"21.2{letter}", answer)

**Ex 21.3 — Tailles de sortie : padding, stride et la hiérarchie 600 → 300 → 100**

In [ ]:
# Ex 21.3: the values you found on paper
answer_21_3a = ...  # a) a whole number
answer_21_3b = ...  # b) a whole number
answer_21_3c = ...  # c) a whole number
answer_21_3d = ...  # d) a whole number
answer_21_3e = ...  # e) a whole number
answer_21_3f = ...  # f) a whole number
answer_21_3g = ...  # g) a whole number
answer_21_3h = ...  # h) a whole number
answer_21_3i = ...  # i) a whole number
answer_21_3j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_21_3a, answer_21_3b, answer_21_3c, answer_21_3d, answer_21_3e, answer_21_3f, answer_21_3g, answer_21_3h, answer_21_3i, answer_21_3j]):
    wb.check(f"21.3{letter}", answer)

**Ex 21.4 — Compter les paramètres : convolution contre couche dense**

In [ ]:
# Ex 21.4: the values you found on paper
answer_21_4a = ...  # a) a whole number
answer_21_4b = ...  # b) a whole number
answer_21_4c = ...  # c) a whole number
answer_21_4d = ...  # d) a whole number
answer_21_4e = ...  # e) a whole number
answer_21_4f = ...  # f) a number (%), 1 decimal
answer_21_4g = ...  # g) a whole number
answer_21_4h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_21_4a, answer_21_4b, answer_21_4c, answer_21_4d, answer_21_4e, answer_21_4f, answer_21_4g, answer_21_4h]):
    wb.check(f"21.4{letter}", answer)

**Ex 21.5 — Champ récepteur effectif d'une pile convolution + pooling**

In [ ]:
# Ex 21.5: the values you found on paper
answer_21_5a = ...  # a) a whole number
answer_21_5b = ...  # b) a whole number
answer_21_5c = ...  # c) a whole number
answer_21_5d = ...  # d) a whole number
answer_21_5e = ...  # e) a whole number
answer_21_5f = ...  # f) a whole number
answer_21_5g = ...  # g) a whole number
answer_21_5h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_21_5a, answer_21_5b, answer_21_5c, answer_21_5d, answer_21_5e, answer_21_5f, answer_21_5g, answer_21_5h]):
    wb.check(f"21.5{letter}", answer)

**Ex 21.6 — Convolution transposée à la main : insérer des zéros puis corréler**

In [ ]:
# Ex 21.6: the values you found on paper
answer_21_6a = ...  # a) a whole number
answer_21_6b = ...  # b) a whole number
answer_21_6c = ...  # c) a whole number
answer_21_6d = ...  # d) a whole number
answer_21_6e = ...  # e) a whole number
answer_21_6f = ...  # f) a whole number
answer_21_6g = ...  # g) a whole number
answer_21_6h = ...  # h) a whole number
answer_21_6i = ...  # i) a whole number
answer_21_6j = ...  # j) a whole number

for letter, answer in zip("abcdefghij", [answer_21_6a, answer_21_6b, answer_21_6c, answer_21_6d, answer_21_6e, answer_21_6f, answer_21_6g, answer_21_6h, answer_21_6i, answer_21_6j]):
    wb.check(f"21.6{letter}", answer)

**Ex 21.8 — Fermi : paramètres, mémoire et calcul de VGG16**

In [ ]:
# Ex 21.8: the values you found on paper
answer_21_8a = ...  # a) a number of millions, 1 decimal
answer_21_8b = ...  # b) a number of billions, 2 decimals
answer_21_8c = ...  # c) a number of billions, 1 decimal
answer_21_8d = ...  # d) a number (%), 1 decimal
answer_21_8e = ...  # e) a number of MB, 1 decimal
answer_21_8f = ...  # f) a number of GB, 2 decimals
answer_21_8g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_21_8a, answer_21_8b, answer_21_8c, answer_21_8d, answer_21_8e, answer_21_8f, answer_21_8g]):
    wb.check(f"21.8{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer la forme de sortie et le nombre de paramètres d'une couche de convolution, avec padding, stride et dilatation ?
2. Sais-tu expliquer le partage de poids, le champ récepteur et ce que font une convolution 1 × 1 et une convolution transposée ?
3. Sais-tu dire comment on regarde ce que voit un filtre, et comment on fabrique un exemple adversarial ?

**Pour aller plus loin** : les notes de CS231n sur les réseaux convolutifs et le guide de Dumoulin et Visin, cités dans la fiche. La suite : le ch. 22, les réseaux récurrents.

*Les exercices de code 21.12 à 21.32 (parties A à D, ta librairie `conv.py` et les CNN PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 21` pour obtenir le notebook complet.*